# Combining & organizing data

**This Jupyter notebook does the following:**

- Imports the prepared rental apartment data
- Joins geocoded addresses and municipality-level data (e.g. population, taxable income) using `.merge()`
- Exports the enriched data to `Data/apartments_data_enriched.csv`
- Demonstrates sorting and aggregating data
- Demonstrates reshaping data using `.stack()`, `.unstack()`, `.melt()` and `.pivot_table()`

## Libraries and settings

In [1]:
# Libraries
import os
import pandas as pd
import numpy as np
import fnmatch

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Show current working directory
print(os.getcwd())

/workspaces/ARM-Stejsnat/Week_09


## Importing the prepared rental apartments data

In [2]:
# Read data to pandas data frame
df_orig = pd.read_csv('./Data/apartments_data_prepared.csv', 
                      sep=',', 
                      encoding='utf-8')

# Copy of data with selected colums
columns = [ 'web-scraper-order', 
            'address_raw',
            'rooms', 
            'area', 
            'luxurious',
            'price',
            'price_per_m2']
df = df_orig[columns]

# Get number of rows and columns
print(df.shape)

# Show first records
df.head(5)

(860, 7)


,web-scraper-order,address_raw,rooms,area,luxurious,price,price_per_m2
0,1693998201-1,"Neuhusstrasse 6, 8630 Rüti ZH, ZH",3.0,49.0,0,1441.0,29.41
1,1693998201-2,"Zürcherstrasse 1, 8173 Neerach, ZH",3.5,65.0,0,1850.0,28.46
2,1693998201-4,"Cramerstrasse 8-12, 8004 Zürich, ZH",2.0,54.0,0,4853.0,89.87
3,1693998201-5,"Rotachstrasse 33, 8003 Zürich, ZH",2.0,49.0,0,4335.0,88.47
4,1693998201-16,"Wolframplatz 1, 8045 Zürich, ZH",2.0,32.0,0,3515.0,109.84


## Combining data from different sources

### Reading rental apartment data with geocoded addresses

In [3]:
# Meaning of variables
# lat: geographical latitude
# lon: geographical longitude
# bfs_number: official municipality id
# bfs_name: official municipality name

# Geocoded data (i.e. data with latitude and longitude)
df_geo = pd.read_csv('./Data/apartments_data_geocoded.csv', 
                     sep=',', 
                     encoding='utf-8')

# Show data
df_geo[['web-scraper-order',
        'address_raw',
        'lat', 
        'lon', 
        'bfs_number', 
        'bfs_name']].head()

,web-scraper-order,address_raw,lat,lon,bfs_number,bfs_name
0,1693998201-1,"Neuhusstrasse 6, 8630 Rüti ZH, ZH",47.252167,8.845798,118,Rüti (ZH)
1,1693998201-2,"Zürcherstrasse 1, 8173 Neerach, ZH",47.513332,8.474851,88,Neerach
2,1693998201-5,"Rotachstrasse 33, 8003 Zürich, ZH",47.370792,8.514748,261,Zürich
3,1693998201-16,"Wolframplatz 1, 8045 Zürich, ZH",47.362282,8.522193,261,Zürich
4,1693998205-25,"Badenerstrasse 67, 8953 Dietikon, ZH",47.407925,8.392561,243,Dietikon


### Join geo-information to rental apartment data using .merge()

In [4]:
df2 = df.merge(df_geo[['web-scraper-order', 
                       'lat', 
                       'lon', 
                       'bfs_number', 
                       'bfs_name']], 
               on="web-scraper-order",
               validate="one_to_one")

# The merge is an inner join, i.e. apartments without geocoded address are dropped
print(f'Apartments before merge: {len(df)}, after merge: {len(df2)}, dropped: {len(df) - len(df2)}')
df2.head()

Apartments before merge: 860, after merge: 815, dropped: 45


,web-scraper-order,address_raw,rooms,area,luxurious,price,price_per_m2,lat,lon,bfs_number,bfs_name
0,1693998201-1,"Neuhusstrasse 6, 8630 Rüti ZH, ZH",3.0,49.0,0,1441.0,29.41,47.252167,8.845798,118,Rüti (ZH)
1,1693998201-2,"Zürcherstrasse 1, 8173 Neerach, ZH",3.5,65.0,0,1850.0,28.46,47.513332,8.474851,88,Neerach
2,1693998201-5,"Rotachstrasse 33, 8003 Zürich, ZH",2.0,49.0,0,4335.0,88.47,47.370792,8.514748,261,Zürich
3,1693998201-16,"Wolframplatz 1, 8045 Zürich, ZH",2.0,32.0,0,3515.0,109.84,47.362282,8.522193,261,Zürich
4,1693998205-25,"Badenerstrasse 67, 8953 Dietikon, ZH",3.5,105.0,0,3200.0,30.48,47.407925,8.392561,243,Dietikon


### Reading municipality-level data

In [5]:
# Meaning of variables:
# bfs_number: official municipality id
# bfs_name: official municipality name
# pop: number of residents (=population)
# pop_dens: population density (pop per km2)
# frg_pct: percentage foreigners
# emp: numer of employees
# mean_taxable_income: mean taxable income

df_municip = pd.read_excel('./Data/municipality_data_with_taxable_income.xlsx', 
                           sheet_name='data_for_import')
df_municip.head(5)

,bfs_number,bfs_name,pop,pop_dens,frg_pct,emp,mean_taxable_income
0,1,Aeugst am Albis,1981,250.442478,14.184755,442.0,105013.570634
1,2,Affoltern am Albis,12303,1161.756374,28.700317,6920.0,71059.805603
2,3,Bonstetten,5572,749.932705,16.564968,1014.0,88927.698145
3,4,Hausen am Albis,3751,275.808824,16.022394,1021.0,86300.455137
4,5,Hedingen,3778,578.560490,16.410799,1478.0,90811.205330


### Join municipality data to rental apartment data using .merge()

In [6]:
# Merge needs a key which must be identical in both data sets (here the key is 'bfs_number')
df3 = df2.merge(df_municip[['bfs_number', 
                            'pop', 
                            'pop_dens', 
                            'frg_pct', 
                            'emp',
                            'mean_taxable_income']], 
                on="bfs_number",
                validate="many_to_one")

# Check whether apartments were dropped by the (inner) join
print(f'Apartments before merge: {len(df2)}, after merge: {len(df3)}, dropped: {len(df2) - len(df3)}')
df3.head(5)

Apartments before merge: 815, after merge: 815, dropped: 0


,web-scraper-order,address_raw,rooms,area,luxurious,price,price_per_m2,lat,lon,bfs_number,bfs_name,pop,pop_dens,frg_pct,emp,mean_taxable_income
0,1693998201-1,"Neuhusstrasse 6, 8630 Rüti ZH, ZH",3.0,49.0,0,1441.0,29.41,47.252167,8.845798,118,Rüti (ZH),12286,1221.272366,24.841283,5053.0,65362.042683
1,1693998201-2,"Zürcherstrasse 1, 8173 Neerach, ZH",3.5,65.0,0,1850.0,28.46,47.513332,8.474851,88,Neerach,3175,525.662252,13.322835,641.0,111422.358722
2,1693998201-5,"Rotachstrasse 33, 8003 Zürich, ZH",2.0,49.0,0,4335.0,88.47,47.370792,8.514748,261,Zürich,420217,4778.994655,32.458468,491193.0,82932.737704
3,1693998201-16,"Wolframplatz 1, 8045 Zürich, ZH",2.0,32.0,0,3515.0,109.84,47.362282,8.522193,261,Zürich,420217,4778.994655,32.458468,491193.0,82932.737704
4,1693998205-25,"Badenerstrasse 67, 8953 Dietikon, ZH",3.5,105.0,0,3200.0,30.48,47.407925,8.392561,243,Dietikon,27746,2970.663812,46.226483,18911.0,64662.737784


### Export data to file

In [7]:
# Remove missing values which may have ocured from the merging process
df3 = df3.dropna()

# Check for missing values per column
print(df3.isna().sum().sum())

# Remove duplicates
df3 = df3.drop_duplicates()

# Check for duplicates
print(df3.duplicated().sum().sum())

# Count number of rows and columns
print(df3.shape)

# Export apartment data to file
df3.to_csv('./Data/apartments_data_enriched.csv',
            sep=',',
            encoding='utf-8',
            index=False)

0
0
(779, 16)


### Sorting data

In [8]:
# Sorting data by 'price' and 'area' with highest price above (ascending=False)
df3.sort_values(by=['price', 'area'], 
                ascending=False).head()

,web-scraper-order,address_raw,rooms,area,luxurious,price,price_per_m2,lat,lon,bfs_number,bfs_name,pop,pop_dens,frg_pct,emp,mean_taxable_income
504,1693998317-653,"8001 Zürich, ZH",4.5,137.0,0,16875.0,123.18,47.374798,8.543448,261,Zürich,420217,4778.994655,32.458468,491193.0,82932.737704
15,1693998205-39,"Binzigerstrasse 52, 8707 Uetikon am See, ZH",6.5,300.0,1,11500.0,38.33,47.267185,8.689932,155,Männedorf,11359,2381.341719,20.380315,4984.0,100477.329317
547,1693998326-702,"Hohenbühlstrasse 7, 8032 Zürich, ZH",5.5,183.0,0,10050.0,54.92,47.366306,8.551867,261,Zürich,420217,4778.994655,32.458468,491193.0,82932.737704
260,1693998265-365,"Gustav-Gull-Platz 4, 8004 Zürich, ZH",4.5,140.0,0,9950.0,71.07,47.379471,8.531120,261,Zürich,420217,4778.994655,32.458468,491193.0,82932.737704
261,1693998265-366,"Engweg 8, 8006 Zürich, ZH",4.5,140.0,0,9170.0,65.50,47.385265,8.537632,261,Zürich,420217,4778.994655,32.458468,491193.0,82932.737704


### Aggregation of data

In [9]:
# Aggregation using .groupby()
df3[['rooms', 'price']].groupby(['rooms']).mean()

,price
rooms,
1.0,1299.509804
1.5,2140.787879
2.0,2155.846154
2.5,2417.581560
3.0,1953.113636
3.5,2650.280000
4.0,2852.357143
4.5,3163.104294
5.0,2881.666667


## Reshaping data

### Reshaping data using .stack() and .unstack()

In [15]:
df_sub = df3[['bfs_name', 'rooms', 'price', 'area']][:5]
print('Original shape')
print(df_sub, '\n')

df_sub_stacked = df_sub.stack()
print('Stacked')
print(df_sub_stacked, '\n')

# Using unstack
print('Unstacked (= back to original shape)')
print(df_sub_stacked.unstack())

Original shape
    bfs_name  rooms   price   area
0  Rüti (ZH)    3.0  1441.0   49.0
1    Neerach    3.5  1850.0   65.0
2     Zürich    2.0  4335.0   49.0
3     Zürich    2.0  3515.0   32.0
4   Dietikon    3.5  3200.0  105.0 

Stacked
0  bfs_name    Rüti (ZH)
   rooms             3.0
   price          1441.0
   area             49.0
1  bfs_name      Neerach
   rooms             3.5
   price          1850.0
   area             65.0
2  bfs_name       Zürich
   rooms             2.0
   price          4335.0
   area             49.0
3  bfs_name       Zürich
   rooms             2.0
   price          3515.0
   area             32.0
4  bfs_name     Dietikon
   rooms             3.5
   price          3200.0
   area            105.0
dtype: object 

Unstacked (= back to original shape)
    bfs_name rooms   price   area
0  Rüti (ZH)   3.0  1441.0   49.0
1    Neerach   3.5  1850.0   65.0
2     Zürich   2.0  4335.0   49.0
3     Zürich   2.0  3515.0   32.0
4   Dietikon   3.5  3200.0  105.0


### Reshaping data using .melt()

In [16]:
df_sub = df3[['rooms', 'price', 'area']][:5]
print('Original shape')
print(df_sub, '\n')

print('Reshaped using .melt()')
print(pd.melt(df_sub, id_vars=['rooms'], value_vars=['price', 'area']))

Original shape
   rooms   price   area
0    3.0  1441.0   49.0
1    3.5  1850.0   65.0
2    2.0  4335.0   49.0
3    2.0  3515.0   32.0
4    3.5  3200.0  105.0 

Reshaped using .melt()
   rooms variable   value
0    3.0    price  1441.0
1    3.5    price  1850.0
2    2.0    price  4335.0
3    2.0    price  3515.0
4    3.5    price  3200.0
5    3.0     area    49.0
6    3.5     area    65.0
7    2.0     area    49.0
8    2.0     area    32.0
9    3.5     area   105.0


### Pivoting data using .pivot_table()

In [17]:
# Using pivot_table to reshape the data and calculate means 
pd.pivot_table(df3[['rooms', 'price', 'area', 'price_per_m2']],
               index=['rooms'],
               values=['price', 'area', 'price_per_m2', 'rooms'],
               aggfunc=(np.mean))

,area,price,price_per_m2,rooms
rooms,,,,
1.0,31.980392,1299.509804,50.250392,1.0
1.5,36.424242,2140.787879,64.503636,1.5
2.0,54.589744,2155.846154,42.528462,2.0
2.5,66.957447,2417.581560,37.331135,2.5
3.0,67.522727,1953.113636,28.779091,3.0
3.5,89.380000,2650.280000,29.734100,3.5
4.0,98.464286,2852.357143,29.646429,4.0
4.5,114.858896,3163.104294,27.066748,4.5
5.0,108.000000,2881.666667,26.376667,5.0


### Relationship between `area` and `price_per_m2`

| Rooms | Mean area (m²) | Mean price (CHF/month) | Mean price per m² (CHF) |
|---:|---:|---:|---:|
| 1 | 31.98 | 1299.51 | 50.25 |
| 1.5 | 36.42 | 2140.79 | 64.50 |
| 2 | 54.59 | 2155.85 | 42.53 |
| 2.5 | 66.96 | 2417.58 | 37.33 |
| 3 | 67.52 | 1953.11 | 28.78 |
| 3.5 | 89.38 | 2650.28 | 29.73 |
| 4 | 98.46 | 2852.36 | 29.65 |
| 4.5 | 114.86 | 3163.10 | 27.07 |
| 5 | 108.00 | 2881.67 | 26.38 |
| 5.5 | 153.81 | 3712.53 | 24.18 |
| 6 | 530.75 | 3450.00 | 16.57 |
| 6.5 | 256.36 | 4607.64 | 22.87 |
| 7 | 175.00 | 6350.00 | 36.33 |
| 7.5 | 205.00 | 4765.00 | 23.27 |
| 8.5 | 210.00 | 8500.00 | 40.48 |

There is a moderate negative relationship between apartment area and `price_per_m2` in this dataset (Pearson correlation ≈ -0.31). Larger apartments tend to have a lower rent per square metre, although this is not true for every listing. One possible explanation is that some costs, such as location-related premiums, are spread across more square metres in larger apartments. The relationship is not exact because location, condition, and other apartment features also influence rent.

### How is the number of apartments distributed in the pivot table?

In [20]:
# Count apartments by number of rooms
apartment_counts = pd.pivot_table(
    df3,
    index=['rooms'],
    values=['web-scraper-order'],
    aggfunc='count'
).rename(columns={'web-scraper-order': 'number_of_apartments'})

display(apartment_counts)

,number_of_apartments
rooms,
1.0,51
1.5,33
2.0,39
2.5,141
3.0,44
3.5,200
4.0,28
4.5,163
5.0,3


The pivot table counts 779 apartments across 15 room categories. Most listings have 3.5 rooms (200 apartments), followed by 4.5 rooms (163) and 2.5 rooms (141). These three categories account for about 65% of all listings. The least common category is 8.5 rooms, with one apartment.

### Table the mean value of rooms, area, prices and price_per_m2 per municipality

In [21]:
municipality_pivot = pd.pivot_table(
    df3,
    index="bfs_name",
    values=["rooms", "area", "price", "price_per_m2"],
    aggfunc="mean"
)

municipality_pivot = municipality_pivot.sort_values(
    by=["price", "area"],
    ascending=[False, False]
)

display(municipality_pivot.round(2))

,area,price,price_per_m2,rooms
bfs_name,,,,
Rüschlikon,144.00,6660.00,46.25,4.50
Männedorf,170.00,5651.33,30.01,4.17
Küsnacht (ZH),136.33,4891.67,35.48,4.50
Nürensdorf,506.50,4500.00,17.60,6.00
Maur,138.50,4250.00,30.44,4.50
...,...,...,...,...
Feuerthalen,81.43,1613.43,19.97,3.64
Dättlikon,90.00,1480.00,16.44,4.00
Weiach,46.00,1420.00,30.87,1.50


### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [18]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.8.0-1070-azure
Datetime: 2026-11-13 09:02:07
Python Version: 3.11.18
-----------------------------------
